##### Copyright 2025 Perceptron AI.


In [ ]:
# Licensed under the MIT License (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://opensource.org/licenses/MIT
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Detect flowers with Qwen 3 VL


<a target="_blank" href="https://colab.research.google.com/github/perceptron-ai-inc/perceptron/blob/main/cookbook/quickstart/quickstart_qwen/quickstart_qwen.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>


This quickstart walks through localizing every flower in a sample frame with the Perceptron Qwen 3 VL model. You'll configure the SDK, run a reusable detection helper, and draw the bounding boxes that the API returns.

![Flower scene sample](https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/quickstart/qwen/flowers.jpg)

You'll learn how to:

- configure the Perceptron client with your API key
- wrap prompts with the `@perceive` decorator for repeated use
- visualize detections by overlaying annotations on the source image

**Note**

Set the `PERCEPTRON_API_KEY` environment variable (or edit the cell below) before running the detection steps. You can also swap in your own frames by pointing `IMAGE_PATH` at another image.

----

## Setup
Install the SDK and prepare the runtime before invoking the API.


### Install dependencies
Install the Perceptron SDK plus helpers for visualization.


In [ ]:
%pip install --upgrade perceptron pillow --quiet

### Configure the Perceptron client
Load your API key, import helpers, and configure the SDK once for the rest of the notebook.


In [ ]:
# Configure the Perceptron client
import os
from pathlib import Path

from IPython.display import Image as IPyImage, display
from PIL import Image, ImageDraw

from cookbook.utils import cookbook_asset
from perceptron import configure, image, perceive, text

api_key = os.getenv("PERCEPTRON_API_KEY", "<Your Perceptron API key>")
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="qwen3-vl-235b-a22b-thinking",
    api_key=api_key,
)

IMAGE_PATH = cookbook_asset("quickstart", "qwen", "flowers.jpg")
ANNOTATED_PATH = Path("flowers_annotated.jpg")

## Detect flowers
Wrap the detection instructions in a reusable helper so you can reuse the same logic for multiple frames.


In [ ]:
# Detect flowers
TARGET_CLASSES = ["flower"]

@perceive(expects="box", allow_multiple=True)
def detect_flowers(frame_path: str):
    scene = image(frame_path)
    friendly_classes = ", ".join(TARGET_CLASSES)
    return scene + text(
        f"Find every {friendly_classes}. Return one bounding box per instance and include a mention attribute."
    )

result = detect_flowers(str(IMAGE_PATH))
print(f"Detected {len(result.points or [])} flowers.")

## Visualize the detections
Draw the bounding boxes on top of the source image and preview the result directly in the notebook.


In [ ]:
# Draw and preview detections
img = Image.open(IMAGE_PATH).convert("RGB")
draw = ImageDraw.Draw(img)

def to_px(point):
    return point.x / 1000 * img.width, point.y / 1000 * img.height

for idx, box in enumerate(result.points or []):
    top_left = to_px(box.top_left)
    bottom_right = to_px(box.bottom_right)
    draw.rectangle([top_left, bottom_right], outline="magenta", width=3)
    label = box.mention or getattr(box, "label", None) or f"flower {idx + 1}"
    text_origin = (top_left[0], max(top_left[1] - 14, 0))
    draw.text(text_origin, label, fill="magenta")

if not (result.points or []):
    print("No flowers detected. Adjust the prompt or try another frame.")


img.save(ANNOTATED_PATH)
display(IPyImage(filename=str(ANNOTATED_PATH)))
print("Annotated image saved to", ANNOTATED_PATH)

## Next steps
- Swap `TARGET_CLASSES` to describe the objects you care about.
- Extend the prompt with additional intents (OCR, captioning, retrieval) for richer workflows.
- Reuse the drawing logic inside your own apps or pipelines once you're happy with the detections.
